# Assignment 3: Plant Disease Detection
## CNN with GPU Acceleration and Hyperparameter Experimentation

In [ ]:
# Install required packages
!pip install tensorflow numpy pandas matplotlib seaborn scikit-learn pillow

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully!")

In [ ]:
# GPU Configuration
print("="*60)
print("GPU CONFIGURATION")
print("="*60)

# Check TensorFlow version
print(f"\nTensorFlow version: {tf.__version__}")

# Check GPU availability
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"\n✓ GPU DETECTED: {len(gpus)} GPU(s) available")
    for i, gpu in enumerate(gpus):
        print(f"  GPU {i}: {gpu}")
    
    try:
        # Enable memory growth to prevent TensorFlow from allocating all GPU memory
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("\n✓ GPU memory growth enabled")
        
        # Set GPU as preferred device
        tf.config.set_soft_device_placement(True)
        print("✓ Soft device placement enabled")
        
    except RuntimeError as e:
        print(f"\n⚠ GPU configuration warning: {e}")
else:
    print("\n⚠ NO GPU DETECTED - Training will use CPU (slower)")
    print("To enable GPU:")
    print("  1. Install CUDA and cuDNN")
    print("  2. Install tensorflow-gpu or tensorflow with GPU support")

# Verify device placement
print(f"\nBuilt with CUDA: {tf.test.is_built_with_cuda()}")
print(f"GPU available: {tf.test.is_gpu_available()}")

print("\n" + "="*60)

In [ ]:
# Dataset configuration
print("\n" + "="*60)
print("DATASET CONFIGURATION")
print("="*60)

# IMPORTANT: Update this path to your dataset location
dataset_base = r'New Plant Diseases Dataset(Augmented)'
train_dir = os.path.join(dataset_base, 'train')
valid_dir = os.path.join(dataset_base, 'valid')
test_dir = os.path.join(dataset_base, 'test')

# Verify directories exist
print("\nChecking dataset directories...")
for dir_name, dir_path in [('Train', train_dir), ('Valid', valid_dir), ('Test', test_dir)]:
    if os.path.exists(dir_path):
        num_classes = len(os.listdir(dir_path))
        print(f"  ✓ {dir_name}: {dir_path} ({num_classes} classes)")
    else:
        print(f"  ✗ {dir_name}: {dir_path} NOT FOUND")

# Image parameters
img_size = 128  # Increased from 64 for better feature learning
batch_size = 32

print(f"\nImage parameters:")
print(f"  Image size: {img_size}x{img_size}")
print(f"  Batch size: {batch_size}")
print(f"  Color mode: RGB (3 channels)")

In [ ]:
# Data augmentation and generators
print("\n" + "="*60)
print("DATA PREPROCESSING & AUGMENTATION")
print("="*60)

print("\nData Augmentation Techniques:")
print("  - Rescaling: 1/255 (normalize to [0,1])")
print("  - Rotation: ±20 degrees")
print("  - Width shift: ±20%")
print("  - Height shift: ±20%")
print("  - Shear: ±20%")
print("  - Zoom: ±20%")
print("  - Horizontal flip: Yes")
print("  - Fill mode: Nearest")

# Training data with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Validation and test data (only rescaling)
valid_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Create generators
print("\nLoading datasets...")

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True
)

valid_generator = valid_datagen.flow_from_directory(
    valid_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

# Get dataset information
num_classes = len(train_generator.class_indices)
class_names = list(train_generator.class_indices.keys())

print(f"\n✓ Dataset loaded successfully!")
print(f"\nDataset Statistics:")
print(f"  Number of classes: {num_classes}")
print(f"  Training samples: {train_generator.samples}")
print(f"  Validation samples: {valid_generator.samples}")
print(f"  Test samples: {test_generator.samples}")
print(f"\nFirst 10 classes: {class_names[:10]}")
if len(class_names) > 10:
    print(f"... and {len(class_names) - 10} more classes")

In [ ]:
# Visualize class distribution
print("\n" + "="*60)
print("CLASS DISTRIBUTION ANALYSIS")
print("="*60)

# Count samples per class in training set
class_counts = {}
for class_name in class_names:
    class_path = os.path.join(train_dir, class_name)
    if os.path.exists(class_path):
        count = len(os.listdir(class_path))
        class_counts[class_name] = count

# Display statistics
counts = list(class_counts.values())
print(f"\nClass balance statistics:")
print(f"  Mean samples per class: {np.mean(counts):.1f}")
print(f"  Std deviation: {np.std(counts):.1f}")
print(f"  Min samples: {np.min(counts)} ({min(class_counts, key=class_counts.get)})")
print(f"  Max samples: {np.max(counts)} ({max(class_counts, key=class_counts.get)})")

# Check if dataset is balanced
cv = np.std(counts) / np.mean(counts)  # Coefficient of variation
if cv < 0.1:
    print(f"\n✓ Dataset is well-balanced (CV: {cv:.3f})")
elif cv < 0.3:
    print(f"\n⚠ Dataset has moderate imbalance (CV: {cv:.3f})")
else:
    print(f"\n⚠ Dataset has significant imbalance (CV: {cv:.3f})")
    print("  Consider using class weights during training")

In [ ]:
# Define hyperparameter configurations
print("\n" + "="*60)
print("HYPERPARAMETER EXPERIMENTATION")
print("="*60)

print("\nExperimenting with THREE key hyperparameters:")
print("  1. Activation Function (ReLU, Tanh)")
print("  2. Learning Rate (0.001, 0.0001, 0.0005)")
print("  3. Dropout Rate (0.3, 0.4, 0.5)")

configs = [
    # Baseline configurations
    {'name': 'Config 1: ReLU, LR=0.001, Dropout=0.3', 'activation': 'relu', 'learning_rate': 0.001, 'dropout': 0.3},
    {'name': 'Config 2: ReLU, LR=0.001, Dropout=0.4', 'activation': 'relu', 'learning_rate': 0.001, 'dropout': 0.4},
    {'name': 'Config 3: ReLU, LR=0.001, Dropout=0.5', 'activation': 'relu', 'learning_rate': 0.001, 'dropout': 0.5},
    
    # Learning rate variations
    {'name': 'Config 4: ReLU, LR=0.0001, Dropout=0.3', 'activation': 'relu', 'learning_rate': 0.0001, 'dropout': 0.3},
    {'name': 'Config 5: ReLU, LR=0.0005, Dropout=0.3', 'activation': 'relu', 'learning_rate': 0.0005, 'dropout': 0.3},
    
    # Different activation
    {'name': 'Config 6: Tanh, LR=0.001, Dropout=0.3', 'activation': 'tanh', 'learning_rate': 0.001, 'dropout': 0.3},
]

print(f"\nTotal configurations to test: {len(configs)}")
print("\nFixed parameters:")
print("  - Architecture: Conv[32→64→128] + Dense[256]")
print("  - Batch Normalization: After each Conv layer")
print("  - Max Pooling: 2x2 after each Conv block")
print("  - Batch size: 32")
print("  - Max epochs: 20 (with early stopping)")
print("  - Early stopping patience: 5 epochs")

print("\nConfigurations:")
for i, config in enumerate(configs, 1):
    print(f"  {i}. {config['name']}")

print("\nEstimated training time:")
if gpus:
    print("  With GPU: ~30-60 minutes for all 6 configs")
    print("  Per config: ~5-10 minutes")
else:
    print("  With CPU: ~3-4 hours for all 6 configs")
    print("  Per config: ~30-40 minutes")
    print("  ⚠ GPU is HIGHLY recommended!")

In [ ]:
# Train models with different configurations
print("\n" + "="*60)
print("MODEL TRAINING")
print("="*60)
print("\nCallbacks:")
print("  - Early Stopping: patience=5, monitor=val_loss")
print("  - ReduceLROnPlateau: factor=0.5, patience=3")
print("\nTraining on GPU..." if gpus else "\nTraining on CPU (this will be slow)...")
print()

results = []

for i, config in enumerate(configs, 1):
    print(f"\n{'='*60}")
    print(f"[{i}/{len(configs)}] Training: {config['name']}")
    print(f"{'='*60}")
    
    # Build CNN model
    model = Sequential([
        # Conv Block 1
        Conv2D(32, 3, activation=config['activation'], padding='same', input_shape=(img_size, img_size, 3)),
        BatchNormalization(),
        MaxPooling2D(2),
        
        # Conv Block 2
        Conv2D(64, 3, activation=config['activation'], padding='same'),
        BatchNormalization(),
        MaxPooling2D(2),
        
        # Conv Block 3
        Conv2D(128, 3, activation=config['activation'], padding='same'),
        BatchNormalization(),
        MaxPooling2D(2),
        
        # Dense layers
        Flatten(),
        Dense(256, activation=config['activation']),
        BatchNormalization(),
        Dropout(config['dropout']),
        Dense(num_classes, activation='softmax')
    ])
    
    # Compile model with specified learning rate
    optimizer = Adam(learning_rate=config['learning_rate'])
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    # Callbacks
    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    )
    
    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
    
    # Train model
    print(f"\nStarting training...")
    history = model.fit(
        train_generator,
        epochs=20,
        validation_data=valid_generator,
        callbacks=[early_stop, reduce_lr],
        verbose=1
    )
    
    epochs_trained = len(history.history['loss'])
    
    # Evaluate on test set
    print(f"\nEvaluating on test set...")
    test_loss, test_accuracy = model.evaluate(test_generator, verbose=0)
    
    # Get predictions for detailed metrics
    test_generator.reset()
    y_pred_prob = model.predict(test_generator, verbose=0)
    y_pred = np.argmax(y_pred_prob, axis=1)
    y_true = test_generator.classes
    
    # Calculate final metrics
    final_train_acc = history.history['accuracy'][-1]
    final_val_acc = history.history['val_accuracy'][-1]
    final_train_loss = history.history['loss'][-1]
    final_val_loss = history.history['val_loss'][-1]
    
    # Store results
    results.append({
        'Config': config['name'],
        'Activation': config['activation'],
        'Learning Rate': config['learning_rate'],
        'Dropout': config['dropout'],
        'Epochs': epochs_trained,
        'Train Accuracy': final_train_acc,
        'Val Accuracy': final_val_acc,
        'Test Accuracy': test_accuracy,
        'Train Loss': final_train_loss,
        'Val Loss': final_val_loss,
        'Test Loss': test_loss
    })
    
    print(f"\n{'='*60}")
    print(f"RESULTS:")
    print(f"  Epochs trained: {epochs_trained}")
    print(f"  Train Accuracy: {final_train_acc:.4f}")
    print(f"  Val Accuracy: {final_val_acc:.4f}")
    print(f"  Test Accuracy: {test_accuracy:.4f}")
    print(f"  Test Loss: {test_loss:.4f}")
    print(f"{'='*60}")

print("\n" + "="*60)
print("✓ ALL CONFIGURATIONS TRAINED SUCCESSFULLY!")
print("="*60)

In [ ]:
# Display comprehensive results
print("\n" + "="*60)
print("COMPREHENSIVE RESULTS TABLE")
print("="*60)

results_df = pd.DataFrame(results)
results_df_sorted = results_df.sort_values('Test Accuracy', ascending=False)

print("\n(Sorted by Test Accuracy - Higher is Better)\n")
display(results_df_sorted[['Config', 'Activation', 'Learning Rate', 'Dropout', 
                           'Val Accuracy', 'Test Accuracy', 'Epochs']])

# Best model
best_config = results_df_sorted.iloc[0]
print("\n" + "="*60)
print("🏆 BEST MODEL")
print("="*60)
print(f"Configuration: {best_config['Config']}")
print(f"\nHyperparameters:")
print(f"  Activation: {best_config['Activation']}")
print(f"  Learning Rate: {best_config['Learning Rate']}")
print(f"  Dropout Rate: {best_config['Dropout']}")
print(f"  Epochs Trained: {int(best_config['Epochs'])}")
print(f"\nPerformance Metrics:")
print(f"  Training Accuracy: {best_config['Train Accuracy']:.4f} ({best_config['Train Accuracy']*100:.2f}%)")
print(f"  Validation Accuracy: {best_config['Val Accuracy']:.4f} ({best_config['Val Accuracy']*100:.2f}%)")
print(f"  Test Accuracy: {best_config['Test Accuracy']:.4f} ({best_config['Test Accuracy']*100:.2f}%)")
print(f"  Test Loss: {best_config['Test Loss']:.4f}")

# Check for overfitting
overfit_gap = best_config['Train Accuracy'] - best_config['Test Accuracy']
if overfit_gap < 0.05:
    print(f"\n✓ Good generalization (overfit gap: {overfit_gap:.4f})")
elif overfit_gap < 0.10:
    print(f"\n⚠ Slight overfitting (overfit gap: {overfit_gap:.4f})")
else:
    print(f"\n⚠ Significant overfitting (overfit gap: {overfit_gap:.4f})")

print("="*60)

In [ ]:
# Hyperparameter impact analysis
print("\n" + "="*60)
print("HYPERPARAMETER IMPACT ANALYSIS")
print("="*60)

# Learning Rate Impact
print("\n1. LEARNING RATE IMPACT:")
print("-" * 60)
lr_analysis = results_df[results_df['Activation'] == 'relu'].groupby('Learning Rate').agg({
    'Test Accuracy': 'mean',
    'Val Accuracy': 'mean',
    'Epochs': 'mean'
}).round(4)
print(lr_analysis)
best_lr = lr_analysis['Test Accuracy'].idxmax()
print(f"\nBest Learning Rate: {best_lr}")

# Dropout Impact
print("\n2. DROPOUT RATE IMPACT:")
print("-" * 60)
dropout_analysis = results_df[results_df['Activation'] == 'relu'].groupby('Dropout').agg({
    'Test Accuracy': 'mean',
    'Val Accuracy': 'mean',
    'Train Accuracy': 'mean'
}).round(4)
print(dropout_analysis)
best_dropout = dropout_analysis['Test Accuracy'].idxmax()
print(f"\nBest Dropout Rate: {best_dropout}")

# Activation Function Impact
print("\n3. ACTIVATION FUNCTION IMPACT:")
print("-" * 60)
activation_analysis = results_df.groupby('Activation').agg({
    'Test Accuracy': 'mean',
    'Val Accuracy': 'mean'
}).round(4)
print(activation_analysis)
best_activation_func = activation_analysis['Test Accuracy'].idxmax()
print(f"\nBest Activation: {best_activation_func}")

In [ ]:
# Visualize results
print("\nGenerating visualizations...")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Learning Rate Impact
lr_data = results_df[results_df['Activation'] == 'relu'].groupby('Learning Rate')['Test Accuracy'].mean()
axes[0, 0].bar(lr_data.index.astype(str), lr_data.values, color='#FF6B6B', edgecolor='black', alpha=0.8)
axes[0, 0].set_xlabel('Learning Rate', fontweight='bold', fontsize=12)
axes[0, 0].set_ylabel('Average Test Accuracy', fontweight='bold', fontsize=12)
axes[0, 0].set_title('Learning Rate Impact on Test Accuracy', fontweight='bold', fontsize=14)
axes[0, 0].set_ylim(0.5, 1.0)
axes[0, 0].grid(axis='y', alpha=0.3)
for i, v in enumerate(lr_data.values):
    axes[0, 0].text(i, v, f'{v:.4f}', ha='center', va='bottom', fontweight='bold')

# 2. Dropout Rate Impact
dropout_data = results_df[results_df['Activation'] == 'relu'].groupby('Dropout')['Test Accuracy'].mean()
axes[0, 1].bar(dropout_data.index.astype(str), dropout_data.values, color='#4ECDC4', edgecolor='black', alpha=0.8)
axes[0, 1].set_xlabel('Dropout Rate', fontweight='bold', fontsize=12)
axes[0, 1].set_ylabel('Average Test Accuracy', fontweight='bold', fontsize=12)
axes[0, 1].set_title('Dropout Rate Impact on Test Accuracy', fontweight='bold', fontsize=14)
axes[0, 1].set_ylim(0.5, 1.0)
axes[0, 1].grid(axis='y', alpha=0.3)
for i, v in enumerate(dropout_data.values):
    axes[0, 1].text(i, v, f'{v:.4f}', ha='center', va='bottom', fontweight='bold')

# 3. Train vs Val vs Test Accuracy (Top 3)
top_3 = results_df_sorted.head(3)
x = np.arange(3)
width = 0.25
axes[1, 0].bar(x - width, top_3['Train Accuracy'], width, label='Train', color='#45B7D1', edgecolor='black')
axes[1, 0].bar(x, top_3['Val Accuracy'], width, label='Validation', color='#95E1D3', edgecolor='black')
axes[1, 0].bar(x + width, top_3['Test Accuracy'], width, label='Test', color='#F38181', edgecolor='black')
axes[1, 0].set_xlabel('Configuration', fontweight='bold', fontsize=12)
axes[1, 0].set_ylabel('Accuracy', fontweight='bold', fontsize=12)
axes[1, 0].set_title('Top 3 Configurations - Train/Val/Test Accuracy', fontweight='bold', fontsize=14)
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(['Config 1', 'Config 2', 'Config 3'])
axes[1, 0].set_ylim(0.5, 1.0)
axes[1, 0].legend()
axes[1, 0].grid(axis='y', alpha=0.3)

# 4. All Configurations Comparison
axes[1, 1].barh(range(len(results_df_sorted)), results_df_sorted['Test Accuracy'], 
                color='#FFA07A', edgecolor='black', alpha=0.8)
axes[1, 1].set_yticks(range(len(results_df_sorted)))
axes[1, 1].set_yticklabels([f"Config {i+1}" for i in range(len(results_df_sorted))])
axes[1, 1].set_xlabel('Test Accuracy', fontweight='bold', fontsize=12)
axes[1, 1].set_title('Test Accuracy Comparison (All Configs)', fontweight='bold', fontsize=14)
axes[1, 1].set_xlim(0.5, 1.0)
axes[1, 1].grid(axis='x', alpha=0.3)
axes[1, 1].invert_yaxis()
for i, v in enumerate(results_df_sorted['Test Accuracy']):
    axes[1, 1].text(v, i, f' {v:.4f}', va='center')

plt.tight_layout()
plt.savefig('assignment3_hyperparameter_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print("\n✓ Analysis saved as 'assignment3_hyperparameter_analysis.png'")

In [ ]:
# Train final best model
print("\n" + "="*60)
print("TRAINING FINAL OPTIMIZED MODEL")
print("="*60)

best_activation_val = best_config['Activation']
best_lr_val = best_config['Learning Rate']
best_dropout_val = best_config['Dropout']

print(f"\nOptimal Hyperparameters:")
print(f"  Activation: {best_activation_val}")
print(f"  Learning Rate: {best_lr_val}")
print(f"  Dropout: {best_dropout_val}")
print("\nTraining final model...")

final_model = Sequential([
    Conv2D(32, 3, activation=best_activation_val, padding='same', input_shape=(img_size, img_size, 3)),
    BatchNormalization(),
    MaxPooling2D(2),
    Conv2D(64, 3, activation=best_activation_val, padding='same'),
    BatchNormalization(),
    MaxPooling2D(2),
    Conv2D(128, 3, activation=best_activation_val, padding='same'),
    BatchNormalization(),
    MaxPooling2D(2),
    Flatten(),
    Dense(256, activation=best_activation_val),
    BatchNormalization(),
    Dropout(best_dropout_val),
    Dense(num_classes, activation='softmax')
])

optimizer = Adam(learning_rate=best_lr_val)
final_model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=0)

history = final_model.fit(
    train_generator,
    epochs=20,
    validation_data=valid_generator,
    callbacks=[early_stop, reduce_lr],
    verbose=0
)

test_loss, test_accuracy = final_model.evaluate(test_generator, verbose=0)

print(f"\n✓ Final model trained successfully!")
print(f"  Epochs: {len(history.history['loss'])}")
print(f"  Final validation accuracy: {history.history['val_accuracy'][-1]:.4f}")
print(f"  Test accuracy: {test_accuracy:.4f}")

In [ ]:
# Test with custom image
print("\n" + "="*60)
print("TESTING WITH CUSTOM IMAGE")
print("="*60)

# Try to use a test image from the test set
test_class = class_names[0]
test_class_path = os.path.join(test_dir, test_class)

if os.path.exists(test_class_path):
    test_images = os.listdir(test_class_path)
    if test_images:
        custom_image_path = os.path.join(test_class_path, test_images[0])
        
        # Load and preprocess image
        img = load_img(custom_image_path, target_size=(img_size, img_size))
        img_array = img_to_array(img) / 255.0
        img_array = np.expand_dims(img_array, axis=0)
        
        # Display image
        plt.figure(figsize=(8, 8))
        plt.imshow(img)
        plt.axis('off')
        plt.title(f'Test Image from Class: {test_class}', fontweight='bold', fontsize=14)
        plt.tight_layout()
        plt.show()
        
        # Predict
        predictions = final_model.predict(img_array, verbose=0)[0]
        predicted_class_idx = np.argmax(predictions)
        predicted_class = class_names[predicted_class_idx]
        confidence = predictions[predicted_class_idx]
        
        print(f"\n{'='*60}")
        print(f"PREDICTION RESULT")
        print(f"{'='*60}")
        print(f"\n🌿 True Label: {test_class}")
        print(f"🎯 Predicted: {predicted_class}")
        print(f"📊 Confidence: {confidence*100:.2f}%")
        print(f"\n{'✓ CORRECT' if predicted_class == test_class else '✗ INCORRECT'}")
        print(f"\n{'='*60}")
        
        # Show top 5 predictions
        print(f"\nTop 5 Predictions:")
        print("-" * 60)
        top_5_idx = np.argsort(predictions)[-5:][::-1]
        for i, idx in enumerate(top_5_idx, 1):
            print(f"  {i}. {class_names[idx]:40s} {predictions[idx]*100:6.2f}%")
    else:
        print("\n⚠ No images found in test directory")
else:
    print(f"\n⚠ Test directory not found: {test_class_path}")

In [ ]:
# Summary
print("\n" + "="*60)
print("ASSIGNMENT 3 COMPLETE ✓")
print("="*60)

print(f"\nDataset:")
print(f"  ✓ Classes: {num_classes}")
print(f"  ✓ Training samples: {train_generator.samples}")
print(f"  ✓ Validation samples: {valid_generator.samples}")
print(f"  ✓ Test samples: {test_generator.samples}")

print(f"\nGPU Status:")
if gpus:
    print(f"  ✓ GPU training enabled ({len(gpus)} GPU(s))")
else:
    print(f"  ⚠ CPU training (consider enabling GPU)")

print(f"\nKey Findings:")
print(f"  ✓ Tested {len(configs)} hyperparameter configurations")
print(f"  ✓ Best Learning Rate: {best_config['Learning Rate']}")
print(f"  ✓ Best Dropout Rate: {best_config['Dropout']}")
print(f"  ✓ Best Activation: {best_config['Activation']}")
print(f"  ✓ Best Test Accuracy: {best_config['Test Accuracy']:.4f} ({best_config['Test Accuracy']*100:.2f}%)")

print(f"\nData Augmentation:")
print(f"  ✓ Rotation, shifts, shear, zoom, flips applied")
print(f"  ✓ Batch normalization after each layer")
print(f"  ✓ Early stopping and LR reduction callbacks")

print("\n" + "="*60)
print("🎉 ALL OBJECTIVES ACHIEVED!")
print("="*60)